In [19]:
import pandas as pd   


submission=pd.read_csv("/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/sample_submission.csv")


submission.iloc[:,0:2]

,id,fake
0,TEST_00000,0
1,TEST_00001,0
2,TEST_00002,0
3,TEST_00003,0
4,TEST_00004,0
...,...,...
49995,TEST_49995,0
49996,TEST_49996,0
49997,TEST_49997,0
49998,TEST_49998,0


In [20]:
test_data=pd.read_csv("/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/test.csv")
test_data

,id,path
0,TEST_00000,./test/TEST_00000.ogg
1,TEST_00001,./test/TEST_00001.ogg
2,TEST_00002,./test/TEST_00002.ogg
3,TEST_00003,./test/TEST_00003.ogg
4,TEST_00004,./test/TEST_00004.ogg
...,...,...
49995,TEST_49995,./test/TEST_49995.ogg
49996,TEST_49996,./test/TEST_49996.ogg
49997,TEST_49997,./test/TEST_49997.ogg
49998,TEST_49998,./test/TEST_49998.ogg


In [21]:
import os
import numpy as np
import pandas as pd
import librosa
import natsort
from sklearn.preprocessing import StandardScaler

test_path='/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/test/'
test_list=natsort.natsorted(os.listdir(test_path))
test_list

['TEST_00000.ogg',
 'TEST_00001.ogg',
 'TEST_00002.ogg',
 'TEST_00003.ogg',
 'TEST_00004.ogg',
 'TEST_00005.ogg',
 'TEST_00006.ogg',
 'TEST_00007.ogg',
 'TEST_00008.ogg',
 'TEST_00009.ogg',
 'TEST_00010.ogg',
 'TEST_00011.ogg',
 'TEST_00012.ogg',
 'TEST_00013.ogg',
 'TEST_00014.ogg',
 'TEST_00015.ogg',
 'TEST_00016.ogg',
 'TEST_00017.ogg',
 'TEST_00018.ogg',
 'TEST_00019.ogg',
 'TEST_00020.ogg',
 'TEST_00021.ogg',
 'TEST_00022.ogg',
 'TEST_00023.ogg',
 'TEST_00024.ogg',
 'TEST_00025.ogg',
 'TEST_00026.ogg',
 'TEST_00027.ogg',
 'TEST_00028.ogg',
 'TEST_00029.ogg',
 'TEST_00030.ogg',
 'TEST_00031.ogg',
 'TEST_00032.ogg',
 'TEST_00033.ogg',
 'TEST_00034.ogg',
 'TEST_00035.ogg',
 'TEST_00036.ogg',
 'TEST_00037.ogg',
 'TEST_00038.ogg',
 'TEST_00039.ogg',
 'TEST_00040.ogg',
 'TEST_00041.ogg',
 'TEST_00042.ogg',
 'TEST_00043.ogg',
 'TEST_00044.ogg',
 'TEST_00045.ogg',
 'TEST_00046.ogg',
 'TEST_00047.ogg',
 'TEST_00048.ogg',
 'TEST_00049.ogg',
 'TEST_00050.ogg',
 'TEST_00051.ogg',
 'TEST_00052

In [22]:
len(test_list)

50000

In [23]:
# 데이터 증강 함수
import random
def augment_audio(y):
    # 시간 축소
    if random.random() < 0.5:
        y = librosa.effects.time_stretch(y, rate=random.uniform(0.8, 1.2))
    # 피치 변경
    if random.random() < 0.5:
        y = librosa.effects.pitch_shift(y, sr=sample_rate, n_steps=random.randint(-3, 3))
    return y

In [24]:
# Constants
duration = 2  # seconds
sample_rate = 32000  # Hz
n_mels = 128
n_mfcc = 40


In [25]:
# Initialize lists for features and labels
test_feature = []

for i in range(len(test_list)):
    x, _ = librosa.load(test_path + test_list[i], sr=sample_rate, duration=duration)
    
    # 데이터 증강
    x = augment_audio(x)
    
    # 노이즈 감소
    x = librosa.effects.preemphasis(x)
    
    # Extract log-mel spectrogram
    S = librosa.feature.melspectrogram(y=x, sr=sample_rate, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Extract MFCC
    mfcc = librosa.feature.mfcc(S=log_S, n_mfcc=n_mfcc)
    
    # Compute delta and delta-delta MFCC
    width = min(7, mfcc.shape[1]) if mfcc.shape[1] >= 3 else 3
    delta_mfcc = librosa.feature.delta(mfcc, width=width)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2, width=width)
    
    # Extract spectral contrast
    spectral_contrast = librosa.feature.spectral_contrast(S=log_S, sr=sample_rate)
    
    # Concatenate features along the MFCC axis
    combined_features = np.concatenate((mfcc, delta_mfcc, delta2_mfcc, spectral_contrast), axis=0)
    
    # Normalize the combined features
    scaler = StandardScaler()
    normalized_features = scaler.fit_transform(combined_features)
    
    # Append to the feature list
    test_feature.append(normalized_features.flatten())


In [26]:
train_feature_df = pd.DataFrame(test_feature)
train_feature_df

,0,1,2,3,4,5,6,7,8,9,...,19929,19930,19931,19932,19933,19934,19935,19936,19937,19938
0,-11.086141,-10.492124,-9.254884,-8.616658,-9.104921,-9.200524,-9.192247,-9.167812,-9.045204,-8.971208,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-11.034095,-10.942673,-10.516183,-10.611385,-10.654711,-10.488430,-10.376245,-10.304779,-10.357569,-10.469435,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-10.832490,-10.639682,-10.396983,-10.512270,-10.607736,-10.781150,-10.922505,-11.007113,-10.788939,-10.533580,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-11.192497,-11.178723,-11.181149,-11.185898,-11.184923,-11.186798,-11.171341,-11.157381,-11.154724,-11.156768,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-11.142870,-11.119996,-11.089479,-11.103902,-11.094253,-10.978234,-10.869828,-10.869337,-10.902168,-10.799550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,-11.084235,-10.987273,-10.659155,-10.417935,-10.259056,-10.279071,-10.426460,-10.544044,-10.610588,-10.754360,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49996,-10.801576,-10.626076,-10.564427,-10.702342,-10.824360,-10.833439,-10.798518,-10.854261,-10.824747,-10.704101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49997,-11.071578,-10.929012,-10.482701,-10.342501,-10.461736,-10.412493,-10.322745,-10.254195,-10.166744,-10.138623,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49998,-11.196484,-11.198876,-11.179616,-11.181653,-11.188553,-11.191065,-11.180360,-11.181861,-11.187355,-11.192219,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
test_x=train_feature_df.iloc[:,0:1016]
test_x

,0,1,2,3,4,5,6,7,8,9,...,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015
0,-11.086141,-10.492124,-9.254884,-8.616658,-9.104921,-9.200524,-9.192247,-9.167812,-9.045204,-8.971208,...,-0.015659,0.010240,-0.452189,-0.853472,-1.384759,-1.744565,-1.796051,-1.699837,-1.580880,-1.776615
1,-11.034095,-10.942673,-10.516183,-10.611385,-10.654711,-10.488430,-10.376245,-10.304779,-10.357569,-10.469435,...,-0.321543,-0.309693,-0.165973,0.119682,0.313084,0.393565,0.578219,0.429423,0.193166,-0.128143
2,-10.832490,-10.639682,-10.396983,-10.512270,-10.607736,-10.781150,-10.922505,-11.007113,-10.788939,-10.533580,...,0.054392,0.105043,-0.470800,-0.774902,-1.164804,-1.263363,-1.257664,-0.955834,-0.754753,-0.540783
3,-11.192497,-11.178723,-11.181149,-11.185898,-11.184923,-11.186798,-11.171341,-11.157381,-11.154724,-11.156768,...,-0.198918,-0.130095,-0.347930,-0.632658,-0.891326,-1.091970,-0.938109,-1.001235,-1.025350,-0.922058
4,-11.142870,-11.119996,-11.089479,-11.103902,-11.094253,-10.978234,-10.869828,-10.869337,-10.902168,-10.799550,...,0.179840,0.482914,0.309993,0.216118,0.013819,-0.045562,-0.021769,-0.116675,-0.302321,-0.390416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,-11.084235,-10.987273,-10.659155,-10.417935,-10.259056,-10.279071,-10.426460,-10.544044,-10.610588,-10.754360,...,0.024270,0.012442,-0.218805,-0.232482,-0.756339,-1.113398,-1.339041,-1.233063,-0.958916,-0.717783
49996,-10.801576,-10.626076,-10.564427,-10.702342,-10.824360,-10.833439,-10.798518,-10.854261,-10.824747,-10.704101,...,-0.757995,-0.862628,-0.742372,-0.622975,-0.532086,-0.675742,-0.713863,-0.725704,-0.915446,-1.015752
49997,-11.071578,-10.929012,-10.482701,-10.342501,-10.461736,-10.412493,-10.322745,-10.254195,-10.166744,-10.138623,...,-0.330139,-0.414816,-0.503765,-0.559572,-0.484673,-0.350102,-0.246171,-0.139877,0.002935,0.117510
49998,-11.196484,-11.198876,-11.179616,-11.181653,-11.188553,-11.191065,-11.180360,-11.181861,-11.187355,-11.192219,...,-0.693058,-0.750233,-0.939009,-1.117691,-1.128797,-1.165021,-1.171280,-1.262508,-1.307942,-1.243322


In [28]:
test_x.isnull().sum()

0       0
1       0
2       0
3       0
4       0
       ..
1011    0
1012    0
1013    0
1014    0
1015    0
Length: 1016, dtype: int64

In [29]:
test_x.to_csv("test data 0713.csv")

In [30]:
test_x

,0,1,2,3,4,5,6,7,8,9,...,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015
0,-11.086141,-10.492124,-9.254884,-8.616658,-9.104921,-9.200524,-9.192247,-9.167812,-9.045204,-8.971208,...,-0.015659,0.010240,-0.452189,-0.853472,-1.384759,-1.744565,-1.796051,-1.699837,-1.580880,-1.776615
1,-11.034095,-10.942673,-10.516183,-10.611385,-10.654711,-10.488430,-10.376245,-10.304779,-10.357569,-10.469435,...,-0.321543,-0.309693,-0.165973,0.119682,0.313084,0.393565,0.578219,0.429423,0.193166,-0.128143
2,-10.832490,-10.639682,-10.396983,-10.512270,-10.607736,-10.781150,-10.922505,-11.007113,-10.788939,-10.533580,...,0.054392,0.105043,-0.470800,-0.774902,-1.164804,-1.263363,-1.257664,-0.955834,-0.754753,-0.540783
3,-11.192497,-11.178723,-11.181149,-11.185898,-11.184923,-11.186798,-11.171341,-11.157381,-11.154724,-11.156768,...,-0.198918,-0.130095,-0.347930,-0.632658,-0.891326,-1.091970,-0.938109,-1.001235,-1.025350,-0.922058
4,-11.142870,-11.119996,-11.089479,-11.103902,-11.094253,-10.978234,-10.869828,-10.869337,-10.902168,-10.799550,...,0.179840,0.482914,0.309993,0.216118,0.013819,-0.045562,-0.021769,-0.116675,-0.302321,-0.390416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,-11.084235,-10.987273,-10.659155,-10.417935,-10.259056,-10.279071,-10.426460,-10.544044,-10.610588,-10.754360,...,0.024270,0.012442,-0.218805,-0.232482,-0.756339,-1.113398,-1.339041,-1.233063,-0.958916,-0.717783
49996,-10.801576,-10.626076,-10.564427,-10.702342,-10.824360,-10.833439,-10.798518,-10.854261,-10.824747,-10.704101,...,-0.757995,-0.862628,-0.742372,-0.622975,-0.532086,-0.675742,-0.713863,-0.725704,-0.915446,-1.015752
49997,-11.071578,-10.929012,-10.482701,-10.342501,-10.461736,-10.412493,-10.322745,-10.254195,-10.166744,-10.138623,...,-0.330139,-0.414816,-0.503765,-0.559572,-0.484673,-0.350102,-0.246171,-0.139877,0.002935,0.117510
49998,-11.196484,-11.198876,-11.179616,-11.181653,-11.188553,-11.191065,-11.180360,-11.181861,-11.187355,-11.192219,...,-0.693058,-0.750233,-0.939009,-1.117691,-1.128797,-1.165021,-1.171280,-1.262508,-1.307942,-1.243322


In [31]:
from sklearn.preprocessing import StandardScaler,RobustScaler

scaler=RobustScaler()

target=pd.DataFrame(scaler.fit_transform(test_x))
target

,0,1,2,3,4,5,6,7,8,9,...,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015
0,-0.249482,1.045361,2.531866,3.694158,2.877866,2.743789,2.770665,2.832298,3.102773,3.301987,...,-0.096855,-0.043422,-0.863455,-1.443658,-2.008711,-2.582524,-2.646238,-2.499254,-2.315486,-2.643667
1,-0.062436,-0.073928,0.418443,0.260902,0.180677,0.471321,0.671134,0.803530,0.724008,0.525458,...,-0.625504,-0.637865,-0.252583,0.395589,0.758492,0.891191,1.188750,0.956026,0.580829,0.055912
2,0.662113,0.678786,0.618173,0.431496,0.262430,-0.045174,-0.297523,-0.449692,-0.057884,0.406582,...,0.024212,0.132724,-0.903176,-1.295163,-1.650220,-1.800739,-1.776621,-1.291915,-0.966748,-0.619837
3,-0.631714,-0.660339,-0.695769,-0.727930,-0.742081,-0.760927,-0.738773,-0.717825,-0.720899,-0.748316,...,-0.413574,-0.304167,-0.640935,-1.026323,-1.204498,-1.522285,-1.260466,-1.365590,-1.408525,-1.244222
4,-0.453359,-0.514446,-0.542168,-0.586803,-0.584284,-0.392922,-0.204115,-0.203847,-0.263121,-0.086316,...,0.241020,0.834819,0.763273,0.577851,0.270739,0.177762,0.219634,0.069838,-0.228106,-0.373593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,-0.242632,-0.184725,0.178879,0.593860,0.869258,0.840728,0.582091,0.376594,0.265390,-0.002569,...,-0.027847,-0.039329,-0.365343,-0.269996,-0.984490,-1.557098,-1.908063,-1.741792,-1.300065,-0.909697
49996,0.773212,0.712587,0.337605,0.104349,-0.114573,-0.137436,-0.077662,-0.176947,-0.122789,0.090572,...,-1.379807,-1.665233,-1.482793,-1.008023,-0.618995,-0.846061,-0.898257,-0.918469,-1.229096,-1.397658
49997,-0.197144,-0.039990,0.474545,0.723696,0.516523,0.605309,0.766004,0.893791,1.069893,1.138522,...,-0.640359,-0.833186,-0.973534,-0.888191,-0.541720,-0.317009,-0.142827,0.032188,0.270256,0.458199
49998,-0.646043,-0.710406,-0.693201,-0.720624,-0.748399,-0.768456,-0.754766,-0.761505,-0.780045,-0.814014,...,-1.267579,-1.456400,-1.902477,-1.943030,-1.591535,-1.640968,-1.637091,-1.789575,-1.869887,-1.770333


In [32]:
X_target = target.astype(np.float32)

In [34]:
import tensorflow as tf
x_target = tf.reshape(X_target,(X_target.shape[0],X_target.shape[1],1))

In [35]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer, Dense, MultiHeadAttention, LayerNormalization
import numpy as np

# 사용자 정의 TransformerBlock 레이어 정의
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="elu"), Dense(embed_dim),]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

In [38]:
# 모델 경로
model_path = '/Users/withmocha/Desktop/DATA/SW DACON(24)/model/0713/voice detection model 0713.keras'  # 저장된 모델 파일 경로

# 저장된 모델 불러오기
model = load_model(model_path, custom_objects={'TransformerBlock': TransformerBlock})

# 모델 요약 출력
model.summary()

/opt/anaconda3/envs/conda_cpu/lib/python3.12/site-packages/keras/src/layers/layer.py:361: UserWarning: `build()` was called on layer 'transformer_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 1016, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 1012, 64)  │        384 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 1012, 64)  │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 1008, 32)  │     10,272 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 1016, 64)  │      8,704 │ input_layer_2[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 1008, 32)  │          0 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 1016, 64)  │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 1004, 16)  │      2,576 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 1016, 32)  │     10,368 │ dropout_9[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 502, 16)   │          0 │ conv1d_5[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 1016, 32)  │          0 │ bidirectional_4[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_1 │ (None, 1016, 1)   │        130 │ input_layer_2[0]… │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 8032)      │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 16)        │      2,624 │ dropout_10[0][0]  │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 1016)      │          0 │ transformer_bloc… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 9064)      │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ bidirectional_5[… │
│                     │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │    580,160 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 64)        │          0 │ dense_7[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │      2,080 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 32)        │          0 │ dense_8[0][0]   

 Total params: 1,851,995 (7.06 MB)

 Trainable params: 617,331 (2.35 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,234,664 (4.71 MB)

In [39]:
y_pred_probs = model.predict(x_target)

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 262s 167ms/step


In [40]:
len(y_pred_probs)

50000

In [47]:
submission=pd.read_csv("/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/sample_submission.csv",index_col=0)


#submission.iloc[:,0:2]
submission

,fake,real
id,,
TEST_00000,0,0
TEST_00001,0,0
TEST_00002,0,0
TEST_00003,0,0
TEST_00004,0,0
...,...,...
TEST_49995,0,0
TEST_49996,0,0
TEST_49997,0,0


In [42]:
y_pred_probs

array([[0.6055235 ],
       [0.7448942 ],
       [0.8072723 ],
       ...,
       [0.24375157],
       [0.9672056 ],
       [0.9054142 ]], dtype=float32)

In [43]:
y_pred_probs[0][0]

0.6055235

In [49]:
submission.columns[0]

'fake'

In [53]:
for i in range(submission.shape[0]):
    real=y_pred_probs[i][0]
    fake=1-real
    submission.iloc[i,1]=fake
    submission.iloc[i,0]=real

In [54]:
submission

,fake,real
id,,
TEST_00000,0.605524,0.394476
TEST_00001,0.744894,0.255106
TEST_00002,0.807272,0.192728
TEST_00003,0.425861,0.574139
TEST_00004,0.499822,0.500178
...,...,...
TEST_49995,0.947240,0.052760
TEST_49996,0.789713,0.210287
TEST_49997,0.243752,0.756248


In [55]:
submission.to_csv("sample_submission.csv")